# Training skeleton cho Table Detection va Structure Recognition

Notebook nay la **khung tham khao** de mo rong sang fine-tune model that tren PubTables-1M/TableBank.

Ghi chu nhanh (de notebook chay duoc neu ban muon thu):
- Can `transformers` va `timm` (Table Transformer backbone).
- Notebook nay khong bat buoc de chay `app.py` / `demo.py`.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class TrainConfig:
    data_dir: Path = Path('data/raw/pubtables1m')
    model_name: str = 'microsoft/table-transformer-detection'
    image_size: int = 1024
    batch_size: int = 2
    epochs: int = 20
    lr: float = 1e-5
    weight_decay: float = 1e-4

cfg = TrainConfig()
cfg

## Dataset loader skeleton

Voi PubTables-1M, can doc image va annotation bbox cua table/row/column/cell. Output nen gom `pixel_values`, `labels` theo format cua Hugging Face Transformers.

In [ ]:
class TableDataset:
    def __init__(self, root: Path, split: str):
        self.root = root
        self.split = split
        self.items = []

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        raise NotImplementedError('Load image, boxes, labels here')

train_ds = TableDataset(cfg.data_dir, 'train')
len(train_ds)

In [ ]:
def build_model(model_name: str):
    from transformers import AutoImageProcessor, TableTransformerForObjectDetection
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = TableTransformerForObjectDetection.from_pretrained(model_name)
    return processor, model

# processor, model = build_model(cfg.model_name)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        batch = {k: v.to(device) if hasattr(v, 'to') else v for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += float(loss.detach().cpu())
    return total_loss / max(1, len(dataloader))